# Chilbolton Climatology — Student 2: Temperature & Relative Humidity

This is **Student 2's notebook** as part of a group project on the climatology of Chilbolton Observatory.
The four students are each analysing a different meteorological variable:
- **Student 1**: Rainfall
- **Student 2 (you)**: Air temperature and relative humidity
- **Student 3**: Surface pressure
- **Student 4**: Wind speed and direction

## Learning Objectives
By the end of this notebook you should be able to:
- Load and quality-control temperature and RH data from NetCDF files
- Convert between physical units (Kelvin to Celsius)
- Calculate climatological statistics and seasonal cycles
- Visualise the joint relationship between temperature and humidity

## Instrument Information
Temperature and relative humidity are measured by the NCAS/STFC temperature-RH sensor at Chilbolton.
- `air_temperature` is stored in **Kelvin (K)**; you will need to convert to Celsius
- `relative_humidity` is stored in **%**
- Each variable has its own QC flag (`qc_flag_air_temperature`, `qc_flag_relative_humidity`)
- QC flag values: 1 = good data, anything else = suspect or bad

## Setup — Run This First

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import netCDF4 as nc4
import numpy as np
import pandas as pd

pd.set_option('display.max_rows', 200)
print('Libraries loaded.')

In [ ]:
def load_temperature_rh(root: str, qc_good_only: bool = True) -> pd.DataFrame:
    """
    Load all temperature-RH NetCDF files from root and return a time-indexed DataFrame.

    Columns returned:
        time     : UTC timestamp (pandas Timestamp)
        temp_c   : air temperature in Celsius
        rh       : relative humidity in %
    """
    files = sorted(Path(root).rglob('*.nc'))
    if not files:
        raise FileNotFoundError(f'No .nc files found under {root}')

    chunks = []
    for f in files:
        with nc4.Dataset(str(f)) as nc:
            unix   = nc.variables['time'][:].data.copy().astype(np.float64)
            temp_k = nc.variables['air_temperature'][:].data.copy().astype(np.float64)
            rh     = nc.variables['relative_humidity'][:].data.copy().astype(np.float64)
            qc_t   = nc.variables['qc_flag_air_temperature'][:].data.copy().astype(np.int8)
            qc_rh  = nc.variables['qc_flag_relative_humidity'][:].data.copy().astype(np.int8)

        temp_c = temp_k - 273.15  # Convert Kelvin -> Celsius
        if qc_good_only:
            temp_c[(qc_t != 0) & (qc_t != 1)] = np.nan
            rh[(qc_rh != 0) & (qc_rh != 1)] = np.nan

        chunks.append(pd.DataFrame({
            'unix':   unix,
            'temp_c': temp_c,
            'rh':     rh,
        }))

    df = pd.concat(chunks, ignore_index=True).sort_values('unix').reset_index(drop=True)
    df['time'] = pd.to_datetime(df['unix'], unit='s', utc=True)
    return df.drop(columns='unix')


print('Helper functions defined.')

## Task 1: Load and Explore the Data

**What to do:**
1. Set `ROOT_PATH` to the temperature-RH data directory
2. Run the cell to load the data into a DataFrame
3. Inspect the DataFrame — how many records are there? What is the date range?

**Questions:**
- What is the time step of the data (how many seconds between measurements)?
- What fraction of temperature readings were flagged as bad quality?
- What is the approximate range of temperatures you see?

In [ ]:
# ===== EDIT THIS =====
ROOT_PATH = '/gws/ssde/j25a/chil_atmos/wx2026/temperature-rh'
QC_GOOD_ONLY = True
PLOT_THEME = 'dark'  # 'light' or 'dark'
# =====================

plt.style.use('dark_background' if PLOT_THEME == 'dark' else 'default')

df = load_temperature_rh(ROOT_PATH, qc_good_only=QC_GOOD_ONLY)

print(f'Loaded {len(df):,} records')
print(f'Date range: {df["time"].min().date()} to {df["time"].max().date()}')
print(f'Time step:  {(df["time"].diff().dt.total_seconds().median()):.0f} s')
print(f'Temperature (°C): min={df["temp_c"].min():.1f}  mean={df["temp_c"].mean():.1f}  max={df["temp_c"].max():.1f}')
print(f'Rel. humidity (%): min={df["rh"].min():.1f}  mean={df["rh"].mean():.1f}  max={df["rh"].max():.1f}')

## Task 2: Monthly Climatology

**What to do:**
Write a function `monthly_clim()` that computes the climatological monthly mean of a variable.
"Climatological monthly mean" means: average all January values across all years into one January
value, all Februarys into one February value, etc.

Your function should:
- Take a `pd.Series` of values and a `pd.Series` of timestamps as input
- Return a DataFrame with columns `month` (1–12) and `mean`, `std`, `count`

**Hints:**
- Use `ts.dt.month` to extract the month number
- Use `.groupby(month).agg(...)` to compute statistics

Then plot the monthly climatology for both temperature and RH on a 2-panel figure.

In [ ]:
MONTH_NAMES = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

def monthly_clim(values: pd.Series, time: pd.Series) -> pd.DataFrame:
    """
    Compute climatological monthly statistics.

    Returns DataFrame with columns: month, mean, std, count
    """
    # TODO: implement
    # Hint: month = time.dt.month
    pass


# Test and plot
temp_clim = monthly_clim(df['temp_c'], df['time'])
rh_clim   = monthly_clim(df['rh'],     df['time'])

print(temp_clim)

# TODO: create a 2-panel figure
# Panel 1: monthly mean temperature (with ±1 std shading)
# Panel 2: monthly mean RH (with ±1 std shading)

## Task 3: Distributions

**What to do:**
Plot histograms (probability distributions) of temperature and relative humidity.

1. Create a histogram of all temperature values (try 100 bins)
2. Create a histogram of all RH values (try 50 bins)
3. Separate the histograms by **season** (DJF, MAM, JJA, SON) and overlay them on the same axes

**Hints:**
- Define a helper `season(month)` that maps month number → season label
- Use `ax.hist(..., density=True, alpha=0.6, label=season)` so all curves are on the same scale
- Use `ax.set_xlabel()`, `ax.legend()`

In [ ]:
def season_label(month: int) -> str:
    """Return season label (DJF, MAM, JJA, SON) for a given month number."""
    # TODO: implement
    # DJF = Dec, Jan, Feb  (months 12, 1, 2)
    # MAM = Mar, Apr, May  (months 3, 4, 5)
    # JJA = Jun, Jul, Aug  (months 6, 7, 8)
    # SON = Sep, Oct, Nov  (months 9, 10, 11)
    pass


# Add season column to df
df['season'] = df['time'].dt.month.map(season_label)

# TODO: plot histograms by season for temperature
# TODO: plot histograms by season for RH

## Task 4: Temperature vs. Relative Humidity

Temperature and relative humidity are physically related — warmer air can hold more water vapour
before it becomes saturated.

**What to do:**
1. Create a 2D density scatter plot of temperature (x-axis) vs. RH (y-axis)
   - Use a 2D histogram (`plt.hist2d`) with 100×100 bins and a logarithmic colour scale
2. Colour points by season (make 4 subplots, one per season)
3. Describe what pattern you see — does RH tend to be higher when temperatures are lower?

**Hints:**
- Drop NaN rows first: `df_clean = df.dropna(subset=['temp_c', 'rh'])`
- `plt.hist2d(x, y, bins=100, norm=mcolors.LogNorm())`
- Add `import matplotlib.colors as mcolors` at the top of the cell

In [ ]:
import matplotlib.colors as mcolors

# TODO: 2D histogram of temp vs RH — all data together

# TODO (optional): 4-panel 2D histogram, one panel per season
# SEASONS = ['DJF', 'MAM', 'JJA', 'SON']
# fig, axes = plt.subplots(2, 2, figsize=(12, 10))
# for ax, s in zip(axes.flat, SEASONS):
#     sub = df_clean[df_clean['season'] == s]
#     ...

## Task 5 (Optional): Daily Cycle

**What to do:**
Investigate whether temperature and RH have a systematic daily (diurnal) cycle.

1. Compute the climatological mean for each **hour of the day** (0–23)
2. Plot mean temperature and RH as a function of hour
3. Separate by season — is the daily cycle stronger in summer or winter?

**Hint:** Use `time.dt.hour` to extract the hour.

In [ ]:
# TODO: compute and plot the daily (diurnal) cycle of temperature and RH

## Reflection Questions

Answer these after completing the tasks above:

1. **Seasonal cycle**: Which month is warmest? Which is coldest? Is this what you would expect for southern England?
2. **RH pattern**: In which season is relative humidity highest? Can you explain why physically?
3. **T–RH relationship**: Describe the scatter plot. Is there a clear relationship? Are there any outliers or clusters?
4. **Data quality**: Were there any gaps in the data? If so, when? How might this affect your conclusions?
5. **Comparison**: How does the temperature climatology at Chilbolton compare to the UK national average (which you can look up)?